# Fase 3: Detector de Estancia Prolongada (id_evento 1007)

**Autor:** Carlos Perez Perez | **MIA 303** | **17/05/2026**

---

A diferencia de los eventos clinicos (caidas, infecciones, errores de medicacion)
que requieren NLP sobre el texto, **"estancia prolongada"** se detecta directamente
de datos estructurados: las fechas de admision y alta en la tabla `admissions`.

Este notebook implementa el detector siguiendo la **estrategia DRG-P75**:
para cada admision, su LOS (length of stay) se compara con el percentil 75 de la
distribucion de LOS dentro de su mismo grupo DRG. Si lo excede, se marca como
estancia prolongada.

Esto cubre el `id_evento 1007` del Anexo 2 (naturaleza GESTION DE LA ORGANIZACION,
impacto 7.35), uno de los eventos administrativos de mas alto impacto.


## 1. Carga de admisiones y DRGs

In [1]:
import duckdb, pandas as pd
from pathlib import Path

PATH_HOSP = Path(r"C:\MIMIC\mimiciv\hosp")
PATH_WORK = Path(r"C:\MIMIC\tesis")

con = duckdb.connect()
admissions = con.execute(f'''
    SELECT subject_id, hadm_id, admittime, dischtime, deathtime,
           hospital_expire_flag, admission_type, discharge_location
    FROM read_csv_auto('{(PATH_HOSP/"admissions.csv.gz").as_posix()}', compression='gzip')
''').df()
drgcodes = con.execute(f'''
    SELECT hadm_id, drg_type, drg_code, description, drg_severity, drg_mortality
    FROM read_csv_auto('{(PATH_HOSP/"drgcodes.csv.gz").as_posix()}', compression='gzip')
''').df()
con.close()

print(f"admissions: {len(admissions):,} hospitalizaciones")
print(f"drgcodes  : {len(drgcodes):,} codigos DRG")
print(f"  tipos DRG: {drgcodes['drg_type'].value_counts().to_dict()}")


admissions: 546,028 hospitalizaciones
drgcodes  : 761,856 codigos DRG
  tipos DRG: {'HCFA': 394925, 'APR': 366931}


## 2. Calcular LOS (Length of Stay) por hospitalizacion

LOS en dias = dischtime - admittime. Se excluyen las admisiones con muerte
intrahospitalaria, porque el "alta" en ese caso es el fallecimiento y no aplica
el concepto de estancia prolongada como problema de gestion.

In [2]:
admissions['admittime'] = pd.to_datetime(admissions['admittime'])
admissions['dischtime'] = pd.to_datetime(admissions['dischtime'])
admissions['los_dias'] = (admissions['dischtime'] - admissions['admittime']).dt.total_seconds() / 86400

print("Distribucion de LOS (en dias, todas las hospitalizaciones):")
print(admissions['los_dias'].describe().round(2).to_string())
print()
print(f"Hospitalizaciones con muerte intrahospitalaria: {admissions['hospital_expire_flag'].sum():,}")

# excluir fallecidos
admisiones_vivos = admissions[admissions['hospital_expire_flag'] == 0].copy()
print(f"Hospitalizaciones consideradas (vivos al alta): {len(admisiones_vivos):,}")


Distribucion de LOS (en dias, todas las hospitalizaciones):
count    546028.00
mean          4.76
std           7.21
min          -0.95
25%           1.13
50%           2.82
75%           5.62
max         515.56

Hospitalizaciones con muerte intrahospitalaria: 11,801
Hospitalizaciones consideradas (vivos al alta): 534,227


## 3. Unir con DRG

Cada hospitalizacion suele tener mas de un DRG asignado (HCFA y APR-DRG son
sistemas distintos). Usamos **APR DRG** porque incluye severidad. Si una
hospitalizacion no tiene APR DRG, usamos HCFA.

In [3]:
drg_apr = drgcodes[drgcodes['drg_type']=='APR'].drop_duplicates('hadm_id', keep='first')
drg_hcfa = drgcodes[drgcodes['drg_type']=='HCFA'].drop_duplicates('hadm_id', keep='first')

# preferir APR, fallback a HCFA
drg_apr_ids = set(drg_apr['hadm_id'])
drg_uso = pd.concat([drg_apr, drg_hcfa[~drg_hcfa['hadm_id'].isin(drg_apr_ids)]])
print(f"DRG por hospitalizacion: {len(drg_uso):,} (APR={len(drg_apr):,} + HCFA fallback={len(drg_uso)-len(drg_apr):,})")

df = admisiones_vivos.merge(drg_uso[['hadm_id','drg_type','drg_code','description','drg_severity']],
                            on='hadm_id', how='left')
sin_drg = df['drg_code'].isna().sum()
print(f"Hospitalizaciones sin DRG asignado: {sin_drg:,} ({100*sin_drg/len(df):.1f}%) - se excluyen")
df = df[df['drg_code'].notna()].copy()
print(f"Cohorte final para el calculo: {len(df):,}")


DRG por hospitalizacion: 394,933 (APR=366,931 + HCFA fallback=28,002)


Hospitalizaciones sin DRG asignado: 150,933 (28.3%) - se excluyen


Cohorte final para el calculo: 383,294


## 4. Calcular el percentil 75 (P75) de LOS por grupo DRG

Para que el P75 sea estadisticamente confiable, exigimos al menos **30 admisiones
por DRG**. Los DRG con menos de 30 casos quedan marcados como NO_EVALUABLE
(muestra insuficiente para definir un umbral).

In [4]:
MIN_CASOS_POR_DRG = 30

# stats por DRG
stats_drg = df.groupby('drg_code').agg(
    n_admisiones=('hadm_id','count'),
    los_mediana=('los_dias','median'),
    los_p75=('los_dias', lambda x: x.quantile(0.75)),
    los_p90=('los_dias', lambda x: x.quantile(0.90)),
).round(2)

drgs_evaluables = stats_drg[stats_drg['n_admisiones'] >= MIN_CASOS_POR_DRG]
print(f"DRGs distintos: {len(stats_drg)}")
print(f"DRGs evaluables (>={MIN_CASOS_POR_DRG} admisiones): {len(drgs_evaluables)} "
      f"({100*len(drgs_evaluables)/len(stats_drg):.1f}%)")
print()
print("Distribucion del P75 entre los DRGs evaluables:")
print(drgs_evaluables['los_p75'].describe().round(2).to_string())


DRGs distintos: 458
DRGs evaluables (>=30 admisiones): 325 (71.0%)

Distribucion del P75 entre los DRGs evaluables:
count    325.00
mean       8.24
std        6.20
min        1.33
25%        4.92
50%        6.83
75%        9.38
max       47.04


## 5. Marcar cada hospitalizacion como AFIRMADO_PROLONGADA o no

Se cruza cada hadm_id con el P75 de su DRG y se calcula el exceso de dias.

In [5]:
df = df.merge(stats_drg[['n_admisiones','los_mediana','los_p75']],
              left_on='drg_code', right_index=True, how='left')

# marca AFIRMADO_PROLONGADA si su LOS supera el P75 de su DRG
df['evento_estancia_prolongada'] = (df['los_dias'] > df['los_p75']) & (df['n_admisiones'] >= MIN_CASOS_POR_DRG)
df['exceso_dias'] = (df['los_dias'] - df['los_p75']).round(2)
df.loc[~df['evento_estancia_prolongada'], 'exceso_dias'] = 0.0
df.loc[df['n_admisiones'] < MIN_CASOS_POR_DRG, 'evento_estancia_prolongada'] = None  # NO_EVALUABLE

n_total = len(df)
n_eval = df['evento_estancia_prolongada'].notna().sum()
n_pos  = (df['evento_estancia_prolongada']==True).sum()
print(f"Hospitalizaciones evaluadas      : {n_eval:,} / {n_total:,}")
print(f"Hospitalizaciones con prolongada : {n_pos:,} ({100*n_pos/n_eval:.1f}% del cohorte evaluable)")
print(f"Exceso medio (cuando ocurre)     : {df.loc[df['evento_estancia_prolongada']==True, 'exceso_dias'].mean():.1f} dias")


Hospitalizaciones evaluadas      : 382,299 / 383,294
Hospitalizaciones con prolongada : 95,651 (25.0% del cohorte evaluable)
Exceso medio (cuando ocurre)     : 5.7 dias


C:\Users\infor\AppData\Local\Temp\ipykernel_40012\3805910376.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[df['n_admisiones'] < MIN_CASOS_POR_DRG, 'evento_estancia_prolongada'] = None  # NO_EVALUABLE


## 6. Top DRGs con mas estancias prolongadas

Esto te muestra en que grupos clinicos se concentra el problema.

In [6]:
top_drgs = (df[df['evento_estancia_prolongada']==True]
            .groupby(['drg_code','description'])
            .agg(n_prolongadas=('hadm_id','count'),
                 exceso_dias_prom=('exceso_dias','mean'))
            .round(2)
            .sort_values('n_prolongadas', ascending=False)
            .head(15))
top_drgs


,,n_prolongadas,exceso_dias_prom
drg_code,description,,
560,VAGINAL DELIVERY,3148,1.53
194,HEART FAILURE,2730,4.94
720,SEPTICEMIA AND DISSEMINATED INFECTIONS,2386,8.42
540,CESAREAN SECTION WITHOUT STERILIZATION,1716,3.97
139,OTHER PNEUMONIA,1509,3.70
175,PERCUTANEOUS CARDIAC INTERVENTION WITHOUT AMI,1393,5.30
254,OTHER DIGESTIVE SYSTEM DIAGNOSES,1386,4.62
463,KIDNEY AND URINARY TRACT INFECTIONS,1321,3.92
383,CELLULITIS AND OTHER SKIN INFECTIONS,1245,3.31


## 7. Guardar la tabla de detecciones

Una fila por hospitalizacion (no por evento detectado, porque a diferencia de los
textos un hadm_id solo puede ser prolongado o no).

In [7]:
out_cols = ['subject_id','hadm_id','admittime','dischtime','los_dias',
            'drg_type','drg_code','description','drg_severity',
            'los_mediana','los_p75','exceso_dias','evento_estancia_prolongada']
out_path = PATH_WORK / "fase3_estancia_prolongada.csv"
df[out_cols].to_csv(out_path, index=False, encoding='utf-8-sig')
print(f"Guardado: {out_path.name}  ({len(df):,} filas)")

# tambien solo las positivas, para integrar con el resto
positivas = df[df['evento_estancia_prolongada']==True][out_cols].copy()
positivas['id_evento']  = 1007
positivas['evento']     = 'Estancia prolongada'
positivas['naturaleza'] = 'GESTION DE LA ORGANIZACION'
positivas['confianza']  = 'alta'  # detector estructural -> alta confianza
positivas['fuente_deteccion'] = 'fase3_los_drg_p75'
out_pos = PATH_WORK / "fase3_estancia_prolongada_positivas.csv"
positivas.to_csv(out_pos, index=False, encoding='utf-8-sig')
print(f"Guardado: {out_pos.name}  ({len(positivas):,} hospitalizaciones positivas)")


Guardado: fase3_estancia_prolongada.csv  (383,294 filas)


Guardado: fase3_estancia_prolongada_positivas.csv  (95,651 hospitalizaciones positivas)


## 8. Integracion con la Matriz GEMSES

Recalculamos la matriz incluyendo el evento 1007 con sus frecuencias reales.
Como las detecciones NegEx son sobre 2,000 epicrisis y este detector trabaja
sobre TODAS las admisiones (~431k), normalizamos al mismo denominador para
que las frecuencias sean comparables.

In [8]:
# cargar matriz GEMSES limpia de NegEx
matriz_nlp = pd.read_csv(PATH_WORK / "fase2_matriz_gemses_limpia.csv", encoding='utf-8-sig')

# cargar anexo2 para impacto del 1007
con = duckdb.connect(r"C:\BASE DATOS\db\gemses.db", read_only=True)
impacto_1007 = con.execute("SELECT impacto FROM anexo2_eventos_adversos WHERE id_evento=1007").fetchone()[0]
con.close()
print(f"Impacto oficial del evento 1007 (Estancia prolongada): G = {impacto_1007}")

# tomar el mismo cohorte de 2,000 hadm_id de la muestra NLP (para que las frecuencias se sumen sobre la misma base)
import csv
det_nlp = pd.read_csv(PATH_WORK / "fase2_detecciones_weak_supervision.csv", encoding='utf-8-sig')
hadm_muestra_nlp = set(det_nlp['hadm_id'].unique())

# de los positivos de prolongada, ver cuantos pertenecen a esa muestra
positivas_en_muestra = positivas[positivas['hadm_id'].isin(hadm_muestra_nlp)]
n_los_en_muestra = len(positivas_en_muestra)
print(f"\nEstancias prolongadas dentro de las 2,000 hospitalizaciones de la muestra NLP: {n_los_en_muestra}")


Impacto oficial del evento 1007 (Estancia prolongada): G = 7.35

Estancias prolongadas dentro de las 2,000 hospitalizaciones de la muestra NLP: 327


## 9. Conclusiones

In [9]:
print('='*70)
print('  RESUMEN DEL DETECTOR DE ESTANCIA PROLONGADA')
print('='*70)
print(f'  Estrategia              : P75 del DRG del paciente')
print(f'  Umbral minimo por DRG   : {MIN_CASOS_POR_DRG} admisiones')
print(f'  Cohorte total           : {len(df):,} hospitalizaciones')
print(f'  Cohorte evaluable       : {n_eval:,} ({100*n_eval/len(df):.1f}%)')
print(f'  Estancias prolongadas   : {n_pos:,} ({100*n_pos/n_eval:.1f}% del evaluable)')
print(f'  Exceso medio cuando +   : {df.loc[df["evento_estancia_prolongada"]==True, "exceso_dias"].mean():.1f} dias')
print(f'  Confianza del detector  : ALTA (datos estructurados, no NLP)')
print()
print(f'  Cobertura nueva en la naturaleza GESTION DE LA ORGANIZACION')
print(f'  (que en la fase NLP estaba practicamente vacia).')


  RESUMEN DEL DETECTOR DE ESTANCIA PROLONGADA
  Estrategia              : P75 del DRG del paciente
  Umbral minimo por DRG   : 30 admisiones
  Cohorte total           : 383,294 hospitalizaciones
  Cohorte evaluable       : 382,299 (99.7%)
  Estancias prolongadas   : 95,651 (25.0% del evaluable)
  Exceso medio cuando +   : 5.7 dias
  Confianza del detector  : ALTA (datos estructurados, no NLP)

  Cobertura nueva en la naturaleza GESTION DE LA ORGANIZACION
  (que en la fase NLP estaba practicamente vacia).


## 10. Limitaciones y notas

- **No es un evento adverso clinico directo**: una estancia prolongada puede deberse
  a un evento adverso (caida, infeccion intrahospitalaria) o a factores no medicos
  (problemas sociales para el alta, demora en transporte). Este detector identifica
  el FENOMENO, no la CAUSA. Combinado con los detectores NLP, sirve para cuantificar
  el impacto de los demas eventos sobre la estancia.

- **DRG no es aleatorio**: pacientes mas graves dentro del mismo DRG tienden a quedarse
  mas. El P75 ya captura esa variabilidad esperada; lo que excede el P75 es la cola
  realmente atipica.

- **No diferencia urgente vs programada**: una cirugia electiva planificada con LOS
  esperado de 2 dias es distinta de una urgencia con LOS variable. Una refinacion futura
  podria estratificar tambien por `admission_type`.

- **El id_evento 1007 esta en la lista REVISAR del Anexo 2**: su Impacto oficial (7.35)
  difiere del calculo por formula (7.95). Mantenemos el valor oficial.
